# 03 — DistilBERT Fine-Tuning for SMS Spam Detection

## 1. Project Objective

The objective of this notebook is to fine-tune a pretrained DistilBERT transformer model to classify SMS messages as either legitimate (ham) or spam.

This experiment builds on the initial exploratory data analysis and the TF-IDF + Logistic Regression baseline developed earlier in the project.Untitled

## 2. Dataset

- **Source:** UCI SMS Spam Collection
- **Task:** Binary text classification
- **Input:** SMS message text
- **Target:** Message label (`ham` or `spam`)
- **Class mapping:** `ham = 0`, `spam = 1`
- **Dataset size after cleaning:** 5,169 messages

## 3. Experimental Workflow

1. Import required libraries and verify the environment.
2. Load the original dataset and reproduce the established cleaning procedure.
3. Split the dataset into training, validation, and test sets using stratified sampling.
4. Encode the target labels as numeric class IDs.
5. Load the pretrained DistilBERT tokenizer.
6. Tokenize the SMS messages and prepare input tensors.
7. Load the pretrained DistilBERT model with a sequence-classification head.
8. Fine-tune the model using the training set.
9. Monitor validation performance and select the best model checkpoint.
10. Evaluate the selected model on the held-out test set.
11. Compare the results with the TF-IDF + Logistic Regression baseline.
12. Save the trained model and tokenizer for later application integration.

## 4. Evaluation Metrics

The model will be evaluated using:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix

Particular attention will be given to spam recall and spam precision because both missed spam messages and legitimate messages incorrectly flagged as spam matter in a practical spam detector.

## 5. Reproducibility and Data Integrity

- Use a fixed random seed for reproducible data splitting and training where applicable.
- Use stratified sampling to preserve class proportions.
- Keep training, validation, and test data separate.
- Use the validation set for model-development decisions.
- Reserve the test set for final evaluation.
- Keep raw and processed datasets out of version control when excluded by `.gitignore`.
- Record model configuration, training settings, and evaluation results.

## 6. Expected Outcome

A fine-tuned DistilBERT spam classifier, a documented evaluation against the classical machine-learning baseline, and saved model artifacts that can later be integrated into a FastAPI application.


In [13]:
# Python standard library
import random
import json
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Deep learning
import torch
from torch.utils.data import Dataset, DataLoader

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Hugging Face Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


In [14]:
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

print("\nAll packages imported successfully!")

PyTorch: 2.14.1+cpu
NumPy: 2.5.2
Pandas: 3.0.5
CUDA build: None
CUDA available: False

All packages imported successfully!


In [20]:
# Locate  raw dataset
data_path = Path("../data/raw/SMSSpamCollection")

# whether the file exists
print("Dataset exists:", data_path.exists())
print("Dataset path:", data_path.resolve())

Dataset exists: True
Dataset path: /home/student/email-spam-detector/data/raw/SMSSpamCollection


In [19]:
df = pd.read_csv(
    data_path,
    sep="\t",
    header=None,
    names=["label", "message"]
)

print("Dataset shape:", df.shape)
print("\nFirst five rows:")
display(df.head())

Dataset shape: (5572, 2)

First five rows:


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [21]:
print("Dataset information:")
df.info()

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Missing values:
label      0
message    0
dtype: int64


In [23]:
# Convert text labels into numerical labels
label_mapping = {
    "ham": 0,
    "spam": 1
}

#creates a new column containing those numerical labels
df["label_id"] = df["label"].map(label_mapping)

# Inspect the conversion
print(df[["label", "label_id"]].head())

# Check for labels that were not mapped
print("\nMissing numerical labels:", df["label_id"].isnull().sum())

  label  label_id
0   ham         0
1   ham         0
2  spam         1
3   ham         0
4   ham         0

Missing numerical labels: 0


In [24]:
X = df["message"]
y = df["label_id"]

print("Messages:", X.shape)
print("Labels:", y.shape)

Messages: (5572,)
Labels: (5572,)


In [27]:
# First split: 70% training, 30% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: divide temporary data equally
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))
print("Total samples:", len(X_train) + len(X_val) + len(X_test))

Training samples: 3900
Validation samples: 836
Testing samples: 836
Total samples: 5572


In [28]:
# Check class proportions in each dataset split
print("Training distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nValidation distribution:")
print(y_val.value_counts(normalize=True).sort_index())

print("\nTesting distribution:")
print(y_test.value_counts(normalize=True).sort_index())

Training distribution:
label_id
0    0.865897
1    0.134103
Name: proportion, dtype: float64

Validation distribution:
label_id
0    0.866029
1    0.133971
Name: proportion, dtype: float64

Testing distribution:
label_id
0    0.866029
1    0.133971
Name: proportion, dtype: float64


In [29]:
# Load the pretrained DistilBERT tokenizer
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

Tokenizer loaded successfully!
Vocabulary size: 30522


In [30]:
# Select the first SMS from our training set
sample_message = X_train.iloc[0]

# Convert the SMS into tokens
sample_tokens = tokenizer.tokenize(sample_message)

# Convert tokens into numerical IDs
sample_token_ids = tokenizer.convert_tokens_to_ids(sample_tokens)

print("Original message:")
print(sample_message)

print("\nTokens:")
print(sample_tokens)

print("\nToken IDs:")
print(sample_token_ids)

Original message:
Goal! Arsenal 4 (Henry, 7 v Liverpool 2 Henry scores with a simple shot from 6 yards from a pass by Bergkamp to give Arsenal a 2 goal margin after 78 mins.

Tokens:
['goal', '!', 'arsenal', '4', '(', 'henry', ',', '7', 'v', 'liverpool', '2', 'henry', 'scores', 'with', 'a', 'simple', 'shot', 'from', '6', 'yards', 'from', 'a', 'pass', 'by', 'berg', '##kam', '##p', 'to', 'give', 'arsenal', 'a', '2', 'goal', 'margin', 'after', '78', 'min', '##s', '.']

Token IDs:
[3125, 999, 9433, 1018, 1006, 2888, 1010, 1021, 1058, 6220, 1016, 2888, 7644, 2007, 1037, 3722, 2915, 2013, 1020, 4210, 2013, 1037, 3413, 2011, 15214, 27052, 2361, 2000, 2507, 9433, 1037, 1016, 3125, 7785, 2044, 6275, 8117, 2015, 1012]
